In [ ]:
import chromadb
from chromadb import DEFAULT_DATABASE
from chromadb import Settings

# For Remote Chroma server:

adminClient= chromadb.AdminClient(Settings(
  chroma_api_impl="chromadb.api.fastapi.FastAPI",
  chroma_server_host="localhost",
  chroma_server_http_port="8000",
))

def get_or_create_tenant_for_user(user_id):
  tenant_id = f"tenant_user:{user_id}"
  try:
    adminClient.get_tenant(tenant_id)
  except Exception as e:
    adminClient.create_tenant(tenant_id)
    adminClient.create_database(DEFAULT_DATABASE, tenant_id)
  return tenant_id, DEFAULT_DATABASE

In [ ]:
import uuid
from PyPDF2 import PdfReader

In [ ]:
reader = PdfReader("sample_docs/MUD LAB QUOTE.pdf")
number_of_pages = len(reader.pages)
page = reader.pages[0]
text = page.extract_text()

In [ ]:
text

In [ ]:
import camelot

def detect_tables(pdf_path):
    tables = camelot.read_pdf(pdf_path, pages='all')
    if tables.n > 0:
        print("Tables found.")
        return tables[1].data
    else:
        print("No tables found.")

pdf_path = "sample_docs/MUD LAB QUOTE.pdf"  # Replace with your PDF file path
extracted_table = detect_tables(pdf_path)
extracted_table

In [ ]:
from img2table.document import PDF
from img2table.ocr import TesseractOCR

# Instantiation of OCR
ocr = TesseractOCR(n_threads=1, lang="eng")

# Instantiation of document, either an image or a PDF
doc = PDF("sample_docs/MUD LAB QUOTE.pdf", 
          pages=[0],
          detect_rotation=False,
          pdf_text_extraction=True)

In [ ]:
# Table extraction
extracted_tables = doc.extract_tables(ocr=ocr,
                                      implicit_rows=True,
                                      borderless_tables=True,
                                      min_confidence=50)

extracted_tables[0][0].df

Table extraction sucks: So, use a combination of camelot/basic table extraction to detect the presence of a table in the pdf. Then, use img2table to extract table using CV

In [ ]:
tenant, db = get_or_create_tenant_for_user("user1")
client = chromadb.HttpClient(host="localhost", port=8000, tenant=tenant, database=db)
collection = client.get_or_create_collection("default")

In [ ]:
collection.peek()

In [ ]:
import pandas as pd

test_excel = pd.read_excel("sample_docs/Oilfield Production Chemicals.xlsx")

"""Given a text in the format of a csv, split the long csv into the sub tables present, accounting for the title of each table. Seperate the title from the table, and promote the correct row as the column headers. Also drop columns that are all empty for each table (e.g, shrink the csv to fit the bounds of each table). Return the result in a csv format. Remove any total or aggregate rows from the table, keep just the raw data. Ensure that each tables dimensions are consistent with each row."""

In [ ]:
import csv


In [ ]:
test_excel.dropna(axis=1, how='all').to_csv("testexcel.csv",quoting=csv.QUOTE_ALL, quotechar="'")

In [ ]:
test_excel_dict = {
    "Cementing Chemicals": "'Chemical Name','Average Prices/Pack','Last supplier ','Usage','Cost of Usage'\n'Cement Dispersant (Kg)','3.81','AlMoghera','187000','712470'\n'Cement Fluid Loss Additive (Kg)','10.95','AlMoghera','127500','1396125'\n'Cement Liquid Extender (Ltr)','0.53','OOIS (Esnad)','438600','232458'\n'Cement Retarder CR 400 (Kg)','3.45','Aubin','61200','211140'\n'Cement Spacer (Kg)','2.83','AlMoghera (Esnad)','187000','529210'\n'Cement Stabilizer (Kg)','12.24','AlMoghera (Esnad)','1700','20808'\n'Cenosphere (Kg)','0.94','OOIS','85000','79900'\n'Defoamer SMLZ-1T (Ltr)','3.61','Shanghai','28220','101874.2'\n'ExpoCem- Expanding Cement Additive (Kg)','3.48','ELKEM','28900','100572'\n'Gas Migration Additive Powder (Kg)','3.56','Basin','34000','121040'\n'Laffsolve-MEB Ethylene (Ltr)','1.56','OOIS','35360','55161.6'\n'Micro Defoamer (Gal)','12.55','OOIS (Esnad)','11220','140811'\n'Micro Silica Powder (Kg)','0.56','AlMoghera','38250','21420.000000000004'\n'Silica flour (Kg)','0.09','OOIS','34000','3060'\n'Surfactant (Ltr)','2.65','AlMoghera (Esnad)','18387.2','48726.08'\n'Cementing Anti Foam (Gal)','13.38','OOIS','','0'\n'Mutual Solvent EGMBE MS-500','7.4','OOIS','','0'\n'Iron Sequestring Agent (ICA-61)','','','',''\n'Ammonium Chloride','','','',''\n'Acetic Acid','','','',''\n'Surfactant Mud Acid (Gal)','21.8246','','',''",

    "Stimulation Chemicals": "'Chemical Name','Average Prices/Pack','current supplier','Usage in 2016 per pack','Cost of Usage'\n'ACETIC ACID  (Gal)','2.6067708333333335','OOIS','44880','116991.875'\n'AMMONIUM BIFLUORID (MT)','1935','OOIS','8.5','16447.5'\n'AMMONIUM CHLORIDE (MT)','320.13','OOIS','68','21768.84'\n'CITRIC ACID ANHYDROUS (Kg)','1.17','OOIS','85000','99450'\n'Hi Temp-Acid Corrosion Inhibitor (AI-600 (Gal)','13.98','Lubrizol','83300','1164534'\n'MUTUAL SOLVENT EGMBE MS-500 (Gal)','4.57','OOIS','26180','119642.6'\n'Non Ionic , Non-Emulsifier (NE-201) (Gal)','11.52','Flotek','63920','736358.4'\n'POTASSIUM CHLORIDE (MT)','429.22','OOIS','850','364837'\n'Santreat MZ CS 30T (Clay Stabilizer) (Gal)','14.1','OOIS','19550','275655'\n'SANTREAT MZ GSX(GLYOXAL BASED)H2S SCAVENGER (Gal)','11.04','OOIS','26180','289027.19999999995'\n'SODA ASH (Kg)','0.43','OOIS','47600','20468'\n'Water Gelling Agent  (LBS)','1.61','OOIS','82280','132470.80000000002'\n'XYLENE (Gal)','3.22','OOIS','61710','198706.2'\n'ACID GELLING AGENT - AGA 500 (Ltr)','3.43','Lubrizol','83300','285719'\n'Acidizing Micellar Solvent (ACID SOL-101 (Gal)','14.73','Flotek (Esnad)','22440','330541.2'\n'All F90 Corrosion Inhibitor Intensifier (Gal)','5.2','Lubrizol','3740','19448'\n'Acid Inhibitor Intensifier (Gal)','14.87','AlMoghera','14960','222455.19999999998'\n'HI TEMP ACID CORROSION INHIBITOR XMI-2104 (Gal)','15.47','AlMoghera','7480','115715.6'\n'Mud Acid Surfactant (Gal)','12.22','AlMoghera','3740','45702.8'\n'ORGANIC IRON REDUCING AGENT ICA-51 (Gal)','8.98','AlMoghera','2244','20151.120000000003'\n'IRON REDUCING AGENT (ICA-720)(DC) kg','9.22','Flotek','0','0'\n'ACID FOAMING AGENT, CAT FOAM (Gal)','16.25','Flotek','0','0'\n'MULTI PURPOSE FOAMING AGENT','9.74','AlMoghera','0','0'\n'AI-600 Acid Corrosion inhibitor (Gal) (discontinued)','15.33','Weatherford','0','0'\n'Acid Emulsifier for Dispersing Acid System EM-10 (Discon)','16.59','Weatherford','0','0'\n'Hydrochloric Acid - 32%HCL  (MT)','115','AlGaith','15300','1759500'\n'Acid Buffering Agent AGA-Buffer-1 WFT-91 (Gal)','8.97','AlMoghera','0','0'\n'Liquid Nitrogen North (Gal)','0.82','Mohsen Hiader','0','0'\n'Liquid Nitrogen South (Gal)','1.03','Mohsen Hiader','0','0'\n'Liquid Nitrogen (Discontinued)','','AlHosni','0','0'\n'Calcium Carbonate - 25 Micron','0.12','OOIS','0','0'\n'Sodium Hypochlorite bleach (Gal)','3.43','OOIS','0','0'\n'HT Corrosion ACI 3004 9c','10.5','OOIS','0','0'\n'H2S Scavenger','','OOIS','0','0'",

    "Potential Products chemicals volumes(old records)": "'Sr#','Product','Annual Volume (kg)','Monthly / tonnes','Oil / water base','Oil based','Diesel %','Diesel qty','Water based','Number of products','Monthly tonnes per product'\n'1','Demulsifier','1487000','123.91666666666667','Oil','1487000','0.3','446100','0','2','61.958333333333336'\n'2','Demulsifier','968000','80.66666666666667','Oil','968000','0.3','290400','0','3','26.88888888888889'\n'3','Corrosion inhibitors','2316000','193','Oil','2316000','0.5','1158000','0','2','96.5'\n'4','Corrosion inhibitors','1260000','105','Water','0','0','0','1260000','2','52.5'\n'5','Biocides','1000000','83.33333333333333','Water','0','0','0','1000000','2','41.666666666666664'\n'6','Scale inhibitor','537000','44.75','Water','0','0','0','537000','1','44.75'\n'7','Scale inhibitor','99000','8.25','Water','0','0','0','99000','1','8.25'\n'8','H2S scavenger','767000','63.916666666666664','Water','0','0','0','767000','1','63.916666666666664'\n'9','Demulsifier PT6791','75600','6.3','Oil','75600','0.3','22680','0','1','6.3'\n'10','Scale inhibitor','36000','3','Water','0','0','0','36000','1','3'\n'11','Biocide MX82275','84000','7','Water','0','0','0','84000','1','7'\n'12','Biocide MX82274','84000','7','Water','0','0','0','84000','1','7'\n'13','Iron sulphide scavenger MX82278','70000','5.833333333333333','Water','0','0','0','70000','1','5.833333333333333'\n'14','Filter aid','36000','3','Water','0','0','0','36000','1','3'\n'15','Calcium Nitrate Solution','900000','75','Water','0','0','0','900000','1','75'\n'16','H2S scavenger','400000','33.333333333333336','Water','0','0','0','400000','1','33.333333333333336'\n'17','Oxygen Scavenger 70% ','50000','4.166666666666667','Water','0','0','0','50000','1','4.166666666666667'\n'18','Demulsifier','960000','80','Oil','960000','0.3','288000','0','1','80'\n'19','Combination CI/Bio/OS','20000','1.6666666666666667','Water','0','0','0','20000','1','1.6666666666666667'\n'20','Biocide','15000','1.25','Water','0','0','0','15000','1','1.25'\n'21','Defoamer, dehydration','65000','5.416666666666667','Oil','65000','0','0','0','1','5.416666666666667'\n'22','Defoamer, MVC','250000','20.833333333333332','Water','0','0','0','250000','1','20.833333333333332'\n'23','Deoiler','40000','3.3333333333333335','Water','0','0','0','40000','1','3.3333333333333335'\n'24','Biocide 1','340000','28.333333333333332','Water','0','0','0','340000','1','28.333333333333332'\n'25','Biocide 2','340000','28.333333333333332','Water','0','0','0','340000','1','28.333333333333332'\n'26','Corrosion inhibitor - oil','231000','19.25','Oil','231000','0.5','115500','0','1','19.25'\n'27','Corrosion inhibitor - gas','35000','2.9166666666666665','Water','0','0','0','35000','1','2.9166666666666665'\n'28','Scale inhibitor','870000','72.5','Water','0','0','0','870000','1','72.5'\n'29','H2S scavenger','650000','54.166666666666664','Water','0','0','0','650000','1','54.166666666666664'\n'30','Demulsifier','720000','60','Oil','720000','0.3','216000','0','1','60'"
}

In [ ]:
import os

In [ ]:
def load_df_from_string(table_string: str, col_delim: str = ","):

  with open("temp.csv", "w+") as f:
    f.write(table_string)

  with open("temp.csv", "r+") as f:
    n = len(f.readline().split(col_delim))

  table_df = pd.read_csv("temp.csv", usecols=range(n),quoting=csv.QUOTE_ALL, quotechar="'")

  os.remove("temp.csv")

  return table_df

In [ ]:
cementing_chemicals = load_df_from_string(test_excel_dict["Cementing Chemicals"])

cementing_chemicals.head(50)

In [ ]:
sim_chemicals = load_df_from_string(test_excel_dict["Stimulation Chemicals"])
sim_chemicals.head(50)

In [ ]:
with open("test.txt", "r+") as f:
  test = f.read()
list_chem_test = [x.strip() for x in test.split("\n")]

list_chem_table = [x.strip() for x in list(sim_chemicals["Chemical Name"])]

for gptd, actual in zip(list_chem_table, list_chem_test):
  if gptd!=actual:
    print(f"gptd: {gptd},  actual: {actual},   {gptd==actual}")

In [ ]:
pp = load_df_from_string(test_excel_dict["Potential Products chemicals volumes(old records)"])
pp.head(50)

New Sheet Test

In [ ]:
import pandas as pd

test_excel = pd.read_excel("sample_docs/Projections for 18 months for chemical production facility.xlsx")
test_excel.dropna(axis=1, how='all').to_csv("testexcel.csv",quoting=csv.QUOTE_ALL, quotechar="'")

In [ ]:
test_excel = test_excel.dropna(axis=1, how='all').dropna(axis=0, how='all')

In [ ]:
import math

def clean_dict(d):
  """
  Recursively remove keys with NaN values or numeric values from nested dictionaries.
  """
  if isinstance(d, dict):
    return {k: clean_dict(v) for k, v in d.items()
      if not (isinstance(v, (int, float)) or pd.isna(v) or (isinstance(v, float) and math.isnan(v)))}
  return d

In [ ]:
excel_dict = test_excel.to_dict()
excel_dict

In [ ]:
clean_dict(excel_dict)

In [ ]:
tables_json = {
    "Production Projections": "'Particulars','Quantity','Selling Price ','July','August','September','October ','November','December','2025-01-01 00:00:00','2025-02-01 00:00:00','2025-03-01 00:00:00','2025-04-01 00:00:00','2025-05-01 00:00:00','2025-06-01 00:00:00','2025-07-01 00:00:00','2025-08-01 00:00:00','2025-09-01 00:00:00','2025-10-01 00:00:00','2025-11-01 00:00:00','2025-12-01 00:00:00'\n'Utilization','','','','','','','','','','','','','','','','','','','','',''\n'Sales( Specify the projected Value)','','','2000000','3000000','3000000','3000000','4000000','4000000','5000000','5000000','6000000','6000000','6000000','6000000','7000000','7000000','8000000','8000000','8000000','8000000'",
    "Raw Materials Used": "'Particulars','Quantity','Cost per Unit','COGS ','July','August','September','October ','November','December','2025-01-01 00:00:00','2025-02-01 00:00:00','2025-03-01 00:00:00','2025-04-01 00:00:00','2025-05-01 00:00:00','2025-06-01 00:00:00','2025-07-01 00:00:00','2025-08-01 00:00:00','2025-09-01 00:00:00','2025-10-01 00:00:00','2025-11-01 00:00:00','2025-12-01 00:00:00'\n'COGS - As per the workings attached','','','1540000','2310000','2310000','2310000','3080000','3080000','3850000','3850000','4620000','4620000','4620000','4620000','5390000','5390000','6160000','6160000','6160000','6160000'\n'COGS','','','1540000','2310000','2310000','2310000','3080000','3080000','3850000','3850000','4620000','4620000','4620000','4620000','5390000','5390000','6160000','6160000','6160000','6160000'",
    "Salaries": "'Desgination','No of Employees','Monthly Salary','SALARIES','July','August','September','October ','November','December','2025-01-01 00:00:00','2025-02-01 00:00:00','2025-03-01 00:00:00','2025-04-01 00:00:00','2025-05-01 00:00:00','2025-06-01 00:00:00','2025-07-01 00:00:00','2025-08-01 00:00:00','2025-09-01 00:00:00','2025-10-01 00:00:00','2025-11-01 00:00:00','2025-12-01 00:00:00'\n'CEO - will be replaced with operating company','1','50000','22068','22068','22068','22068','22068','22068','22068','22068','22068','22068','22068','22068','22068','22068','22068','22068','22068','22068','22068'\n'COO - will be replaced with operating company','1','50000','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0'\n'CTO - will be replaced with operating company','1','50000','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0','0'\n'PRODUCTION MANAGER','1','16000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000','10000'",
    "Wages": "'Desgination','No of Employees','Monthly Wages','WAGES','July','August','September','October ','November','December','2025-01-01 00:00:00','2025-02-01 00:00:00','2025-03-01 00:00:00','2025-04-01 00:00:00','2025-05-01 00:00:00','2025-06-01 00:00:00','2025-07-01 00:00:00','2025-08-01 00:00:00','2025-09-01 00:00:00','2025-10-01 00:00:00','2025-11-01 00:00:00','2025-12-01 00:00:00'\n'HELPER','2','2500','5000','7500','7500','7500','10000','10000','12500','12500','15000','15000','15000','15000','17500','17500','20000','20000','20000','20000','20000'",
    "Utilities": "'Particulars','Quantity','Cost','UTILITIES','July','August','September','October ','November','December','2025-01-01 00:00:00','2025-02-01 00:00:00','2025-03-01 00:00:00','2025-04-01 00:00:00','2025-05-01 00:00:00','2025-06-01 00:00:00','2025-07-01 00:00:00','2025-08-01 00:00:00','2025-09-01 00:00:00','2025-10-01 00:00:00','2025-11-01 00:00:00','2025-12-01 00:00:00'\n'DEWA','1','','6000','15000','15000','15000','20000','20000','25000','25000','30000','30000','30000','30000','35000','35000','40000','40000','40000','40000'"
}
tables_df: dict[str, pd.DataFrame] = {}


In [ ]:
for key, value in tables_json.items():
  tables_df[key] = load_df_from_string(value)

In [ ]:
testing = {"sa": 52, "sagas": "fsfs"}

In [ ]:
"eare/greaern".split("/")[-1]

In [ ]:
print(type(str(testing)))